<a href="https://colab.research.google.com/github/NBall65097/Fantasy-Story-Weaver/blob/main/Fantasy_Story_Weaver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers==0.0.28.post2 trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xwxqw73m/unsloth_acefa34ec1cc4accbda272a8504610fb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xwxqw73m/unsloth_acefa34ec1cc4accbda272a8504610fb
  Resolved https://github.com/unslothai/unsloth.git to commit 997f1a7ce5fb7a0319c2b8abe0e7af02e2160efe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 22.1 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('HuggingFaceWrite')
login(token) # Connect to HuggingFace account

In [3]:
import datasets # datasets function for importing data

In [5]:
# # This was used to debug issues with the AI-generated training data formatting.
# import json

# file_path = '/content/data/FSTData.jsonl'

# with open(file_path, 'r', encoding='utf-8') as f:
#     for i, line in enumerate(f, 1):
#         line = line.strip()
#         if not line:
#             continue
#         try:
#             json.loads(line)
#         except json.JSONDecodeError as e:
#             print(f"First error at row {i}: {e}")
#             print(f"Problem line (first 200 chars): {line[:200]}...")
#             print(f"Full line length: {len(line)}")
#             break
#     else:
#         print("All lines appear valid!")

In [6]:
#filepath = "/content/data/FSTData.jsonl"
#data = datasets.load_dataset("json",data_files=filepath,split='train')
#data.push_to_hub('NBall65097/fantasy-storyweaver-data') # Save dataset to HuggingFace

In [7]:
#data = data.train_test_split(test_size=0.1, train_size=0.9)
#traindata, testdata = data["train"], data["test"]

In [4]:
import torch
from datasets import load_dataset
import unsloth
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "microsoft/Phi-3.5-mini-instruct",
    dtype=None,  # auto
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
import json
dataset = load_dataset('NBall65097/fantasy-storyweaver-data',split="train")

README.md:   0%|          | 0.00/343 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/125k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/165 [00:00<?, ? examples/s]

In [12]:
from unsloth.chat_templates import apply_chat_template

In [13]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="phi-3")

def format_phi35_data(examples):
  formatted_texts = []
  for msg_list in examples["messages"]:
    text = tokenizer.apply_chat_template(
        msg_list,
        tokenize=False,
        add_generation_prompt=False,
    )
    formatted_texts.append(text)
  return {"text":formatted_texts}

In [14]:
dataset = dataset.map(
    format_phi35_data,
    batched=True,
    batch_size=1000,          # adjust based on RAM
    num_proc=2,               # or 4 if you have enough CPU cores
    desc="Formatting Phi-3.5 chat examples",
)

Formatting Phi-3.5 chat examples (num_proc=2):   0%|          | 0/165 [00:00<?, ? examples/s]

In [15]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="/content/output",
        num_train_epochs=2,
        learning_rate=2e-4,
        max_seq_length=2048,
        dataset_text_field="text",  # or use formatting_func for chat template
        packing=True,  # packs multiple examples → huge speed win
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        fp16=True,
        gradient_checkpointing="unsloth",
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        #save_steps=200,
        report_to="none",
    )  # TrainingArguments: 1–3 epochs, lr=2e-4, batch=4–8 (gradient accum=4), warmup, etc.
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/165 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/165 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51 | Num Epochs = 2 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss


Step,Training Loss


TrainOutput(global_step=8, training_loss=2.266510248184204, metrics={'train_runtime': 429.5718, 'train_samples_per_second': 0.237, 'train_steps_per_second': 0.019, 'total_flos': 3913699205652480.0, 'train_loss': 2.266510248184204, 'epoch': 2.0})

In [17]:
model.save_pretrained("fantasy-weaver-lora")          # saves adapter only (~50–200 MB)
tokenizer.save_pretrained("fantasy-weaver-lora")

('fantasy-weaver-lora/tokenizer_config.json',
 'fantasy-weaver-lora/chat_template.jinja',
 'fantasy-weaver-lora/tokenizer.json')